### **Import thư viện**

In [ ]:
import os

# Tắt backend CPU oneDNN nếu cần
os.environ["FLAGS_use_mkldnn"] = "0"

import time
from pathlib import Path

import fitz
import paddle
from paddleocr import PaddleOCR

c:\2_workspace\GSOFT\Azure-Devops-AI\GSOFT_AI_Extension_for_Azure_DevOps\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Kiểm tra xem có dùng CUDA không?**

In [2]:
print("Paddle version:", paddle.__version__)
print("CUDA:", paddle.is_compiled_with_cuda())
print("GPU count:", paddle.device.cuda.device_count())

if paddle.is_compiled_with_cuda():
    paddle.set_device("gpu:0")
    print("Current device:", paddle.device.get_device())
else:
    paddle.set_device("cpu")
    print("Current device:", paddle.device.get_device())

Paddle version: 3.2.0
CUDA: False
GPU count: 0
Current device: cpu


In [ ]:
!nvidia-smi

### **Inititalize OCR**

In [17]:
ocr = PaddleOCR(
    use_angle_cls=True,
    lang="vi",
)

C:\Users\khanh\AppData\Local\Temp\ipykernel_15584\2110002069.py:1: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\khanh\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\khanh\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\khanh\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete

### **Set up Path PDF**

In [4]:
PROJECT_ROOT = Path.cwd().parent
PDF_PATH = PROJECT_ROOT / "storage" / "projects" / "proj-namabank-01" / "documents" / "contracts" / "hd cc phan mem - so 03.2026.hdkt.gsoft-nhna- nam á (ngày 10032026)-07-05-26 10-36-56.pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {PDF_PATH}")

print("Input file:", PDF_PATH)
print("File size:", round(PDF_PATH.stat().st_size / 1024, 2), "KB")

doc = fitz.open(str(PDF_PATH))

print(f"Total pages: {len(doc)}")

Input file: c:\2_workspace\GSOFT\Azure-Devops-AI\GSOFT_AI_Extension_for_Azure_DevOps\backend\storage\projects\proj-namabank-01\documents\contracts\hd cc phan mem - so 03.2026.hdkt.gsoft-nhna- nam á (ngày 10032026)-07-05-26 10-36-56.pdf
File size: 8497.81 KB
Total pages: 31


### **Render page 17 to image**

In [12]:
page_number = 17
page = doc[page_number - 1]

# Kích thước trang PDF
rect = page.rect

# Cắt bớt 4 cạnh
left = rect.width * 0.1
top = rect.height * 0.1
right = rect.width * 0.92
bottom = rect.height * 0.9

crop_rect = fitz.Rect(
    left,
    top,
    right,
    bottom
)

pix = page.get_pixmap(
    dpi=400,
    alpha=False,
    clip=crop_rect
)

image_path = Path(
    f"page_{page_number}_cropped.png"
)

pix.save(str(image_path))

print(f"Saved cropped image: {image_path}")

Saved cropped image: page_17_cropped.png


### **OCR & Print results**

In [19]:

start = time.perf_counter()

result = ocr.predict("page_17_cropped.png")

elapsed = time.perf_counter() - start

print("\n" + "=" * 100)
print(f" OCR RESULT - PAGE {page_number}")
print("=" * 100)
print(f"⏱️  OCR Time : {elapsed:.2f}s")
print(f"🖥️  Device   : {paddle.device.get_device()}")
print("=" * 100)

for page_result in result:
    if page_result is None:
        continue

    # Lấy trực tiếp từ page_result (như một dict thông thường)
    rec_texts = page_result.get("rec_texts", [])
    rec_scores = page_result.get("rec_scores", [])

    print(f"\n{'No.':<5} {'Score':<10} Text")
    print("-" * 100)

    for i, (text, score) in enumerate(
        zip(rec_texts, rec_scores),
        start=1
    ):
        print(f"{i:<5} {score:<10.3f} {text}")


print("\n" + "=" * 100)
print(" OCR COMPLETED")
print("=" * 100)


 OCR RESULT - PAGE 17
⏱️  OCR Time : 18.77s
🖥️  Device   : cpu

No.   Score      Text
----------------------------------------------------------------------------------------------------
1     0.924      DANH SÁCH PHÂN HĘ VÀ CHÚC NĂNG PHÀN MĒM
2     0.986      I.
3     0.958      MÔ TÀ/ GHI CHÚ
4     1.000      STT
5     0.939      TÊN CHÚC NĂNG
6     0.980      I
7     0.929      QUÀN TRI HĘ THÓNG
8     0.935      QUÅN LÝ DANH MC
9     0.867      Ⅱ
10    0.943      QUÀN LÝ KHO TM
11    0.813      III
12    0.970      QUÁN LÝ NHP KHO TM CHÚNG TÙ (TP)
13    1.000      1
14    0.973      - ĐVKD đăng nhp phn mm bng user → chn Khu
15    0.985      vc → Chn nhp kho (bung ra trưòng nhp liu Biên
16    1.000      bn bàn giao)
17    0.976      ĐVKD lp biên bn bàn giao chng t k toán theo
18    0.988      mu ca đon v
19    0.999      S có các ct đ chn:
20    0.975      - STT: (h thng t nhy s tù 1 đn n)
21    0.996      - Ngày giao: (h thng t nhy ngày hin hành)
22    0.953      - S lưng: (ĐVKD nh

In [ ]:
import cv2
import numpy as np


def remove_red_stamps(image_path: str, output_path: str):
    img = cv2.imread(image_path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Dải màu đỏ trong HSV (đỏ nằm ở 2 đầu vòng Hue)
    lower_red1 = np.array([0, 70, 50])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([170, 70, 50])
    upper_red2 = np.array([180, 255, 255])
    mask = cv2.inRange(hsv, lower_red1, upper_red1) | cv2.inRange(hsv, lower_red2, upper_red2)

    # Thay vùng đỏ (mộc/chữ ký đỏ) bằng trắng — coi như xoá nhiễu
    img[mask > 0] = [255, 255, 255]
    cv2.imwrite(output_path, img)

remove_red_stamps("page_17.png", "page_17_clean.png")

In [ ]:
input_path = str(image_path)

start = time.perf_counter()

output = pipeline.predict(input_path)

elapsed = time.perf_counter() - start

print(f"PP-StructureV3 time: {elapsed:.2f}s")

for res in output:
    if res is None:
        continue

    res.print()

In [ ]:
output_dir = Path("output_ppstructure")
output_dir.mkdir(exist_ok=True)

for res in output:
    if res is None:
        continue

    res.save_to_json(
        save_path=str(output_dir)
    )

    res.save_to_markdown(
        save_path=str(output_dir)
    )